In [4]:
import pandas as pd
import numpy as np
from DATA.stock_invest_function import *
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

# 한글 폰트 설정
def setup_korean_font():
    """운영체제에 맞는 한글 폰트 설정"""
    system = platform.system()

    try:
        if system == "Windows":
            # Windows의 경우
            font_candidates = ['Malgun Gothic', 'Microsoft YaHei', 'SimHei']
        elif system == "Darwin":  # macOS
            font_candidates = ['AppleGothic', 'Apple SD Gothic Neo']
        else:  # Linux
            font_candidates = ['DejaVu Sans', 'Liberation Sans']

        # 사용 가능한 폰트 찾기
        available_fonts = [f.name for f in fm.fontManager.ttflist]

        for font in font_candidates:
            if font in available_fonts:
                plt.rcParams['font.family'] = font
                print(f"한글 폰트 설정 완료: {font}")
                break
        else:
            # 대안: 유니코드를 지원하는 기본 폰트
            plt.rcParams['font.family'] = 'DejaVu Sans'
            print("기본 유니코드 폰트 사용")

        # 마이너스 기호 깨짐 방지
        plt.rcParams['axes.unicode_minus'] = False

    except Exception as e:
        print(f"폰트 설정 오류: {e}")
        # 안전한 기본 설정
        plt.rcParams['font.family'] = 'DejaVu Sans'
        plt.rcParams['axes.unicode_minus'] = False

# 한글 폰트 설정 실행
setup_korean_font()

# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': '3307',
    'database': 'investar'
}

# 설정값들 (선택사항 - 기본 분석용)
# target_ticker = 'META'
# target_item = 'sale'

# 또는 여러 분석을 한 번에 하고 싶다면:
analysis_list = [
    ('META', 'sale'),
    ('AAPL', 'sale'),
    ('GOOGL', 'netincome'),
    ('TSLA', 'sale')
]

# DB 엔진 생성 함수
def make_engine(db_info: dict):
    return create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}",
        pool_pre_ping=True
    )

# ticker 데이터 조회 함수
def fetch_ticker_data(engine, table_name: str, ticker: str, ticker_col: str = "ticker") -> pd.DataFrame:
    sql = text(f"SELECT * FROM `{table_name}` WHERE `{ticker_col}` = :ticker")
    with engine.connect() as conn:
        df = pd.read_sql(sql, conn, params={"ticker": ticker})
    return df

# 성장률 계산 함수
def calculate_growth_rates(df):
    """분기별, 연간 성장률을 계산하는 함수"""
    growth_df = df.copy()
    growth_df = growth_df.sort_values('date').reset_index(drop=True)

    # 분기별 성장률 (QoQ: Quarter over Quarter)
    growth_df['qoq_growth_rate'] = growth_df['value'].pct_change() * 100

    # 연간 성장률 (YoY: Year over Year) - 4분기 전과 비교
    growth_df['yoy_growth_rate'] = growth_df['value'].pct_change(periods=4) * 100

    return growth_df

def get_company_name(ticker):
    """티커를 그대로 반환"""
    return ticker

# ==============================================
# 메인 실행 부분
# ==============================================

print("데이터 로딩 중...")
engine = make_engine(db_info)

# 1. 한 번만 전체 데이터 로드 (시간 절약을 위해)
print("전체 미국 기업 재무정보 로딩 중... (처음 한 번만)")
historical_df_full = fetch_table_data(db_info, 'US_IS_from_FMP')
print(f"전체 Historical 데이터 로드 완료: {len(historical_df_full)} 행")

# 2. 전체 예측 데이터도 한 번만 로드 (모든 티커)
print("전체 예측 데이터 로딩 중... (모든 티커)")
try:
    # 모든 티커의 예측 데이터를 한 번에 로드
    sql = text("SELECT * FROM us_fs_forecast_data")
    with engine.connect() as conn:
        forecast_df_full = pd.read_sql(sql, conn)
    print(f"전체 Forecast 데이터 로드 완료: {len(forecast_df_full)} 행")
except Exception as e:
    print(f"예측 데이터 로드 오류: {e}")
    forecast_df_full = pd.DataFrame()

def analyze_ticker_item(ticker, item, historical_df_full, forecast_df_full):
    """특정 티커와 아이템에 대한 분석을 수행하는 함수"""

    print(f"\n========== {ticker} - {item} 분석 ==========")

    # Historical 데이터에서 해당 티커와 아이템 추출
    extracted_df = historical_df_full[(historical_df_full['ticker'] == ticker) &
                                     (historical_df_full['fs_item'] == item)]
    extracted_resize_df = extracted_df[['date', 'ticker', 'fs_item', 'value']]
    historical_fs_df = extracted_resize_df.rename(columns={'fs_item': 'item'})

    # Forecast 데이터에서 해당 티커와 아이템 추출
    df_filtered = forecast_df_full[(forecast_df_full['ticker'] == ticker) &
                                  (forecast_df_full["target_col"] == item)].copy()

    if len(df_filtered) > 0:
        df_filtered["prediction_date"] = pd.to_datetime(df_filtered["prediction_date"])
        forecast_fs_df = df_filtered[
            df_filtered["prediction_date"] == df_filtered["prediction_date"].max()
        ][['forecast_date', 'ticker', 'target_col', 'forecast_value']]
        forecast_fs_df = forecast_fs_df.rename(columns={'forecast_date': 'date',
                                                       'target_col': 'item',
                                                       'forecast_value': 'value'})
    else:
        forecast_fs_df = pd.DataFrame()

    # 데이터 결합
    df = pd.concat([historical_fs_df, forecast_fs_df], ignore_index=True)

    if len(df) == 0:
        print(f"경고: {ticker} - {item}에 대한 데이터가 없습니다.")
        return None, None

    # 날짜 컬럼을 datetime으로 변환
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    # 성장률 계산
    growth_df = calculate_growth_rates(df)

    print(f"데이터 기간: {df['date'].min().strftime('%Y-%m-%d')} ~ {df['date'].max().strftime('%Y-%m-%d')}")
    print(f"총 데이터 포인트: {len(df)}개")

    return df, growth_df


한글 폰트 설정 완료: Malgun Gothic
데이터 로딩 중...
전체 미국 기업 재무정보 로딩 중... (처음 한 번만)
❌ 데이터 조회 실패: (pymysql.err.OperationalError) (2013, 'Lost connection to MySQL server during query')
[SQL: SELECT * FROM US_IS_from_FMP]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
전체 Historical 데이터 로드 완료: 0 행
전체 예측 데이터 로딩 중... (모든 티커)
예측 데이터 로드 오류: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '192.168.0.230' ([WinError 10061] 대상 컴퓨터에서 연결을 거부했으므로 연결하지 못했습니다)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [5]:
# 기본 분석 실행 (원하는 회사/항목으로 변경)
target_ticker = 'NVDA'  # 원하는 티커로 변경
target_item = 'sale'    # 원하는 항목으로 변경

df, growth_df = analyze_ticker_item(target_ticker, target_item, historical_df_full, forecast_df_full)

# 또는 여러 분석을 한 번에 실행하려면:
# for ticker, item in analysis_list:
#     df, growth_df = analyze_ticker_item(ticker, item, historical_df_full, forecast_df_full)
#     if df is not None:
#         print(f"\\n{ticker} - {item} 분석 완료!")
#     print("="*50)

if df is not None:
    # Historical과 Forecast 데이터 구분점 찾기
    historical_data = growth_df[growth_df['date'] <= '2025-06-30']  # 2025년 2분기까지를 historical로 가정
    forecast_data = growth_df[growth_df['date'] > '2025-06-30']     # 2025년 3분기부터를 forecast로 가정

    # 성장률 정보 출력
    print(f"\n========== 성장률 분석 ==========")

    # 마지막 historical 데이터 표시
    if len(historical_data) > 0:
        last_historical = historical_data.iloc[-1]
        last_hist_date = last_historical['date'].strftime('%Y-%m-%d')
        last_hist_value = last_historical['value'] / 1e9
        print(f"마지막 Historical 데이터: {last_hist_date} - ${last_hist_value:.1f}B")

    # Forecast 데이터 성장률 정보 (2025년 3분기부터)
    if len(forecast_data) > 0:
        print(f"\nForecast 데이터 성장률 (2025년 3분기 이후):")
        forecast_growth = forecast_data[['date', 'value', 'qoq_growth_rate', 'yoy_growth_rate']].copy()
        forecast_growth['value_billions'] = forecast_growth['value'] / 1e9

        for idx, row in forecast_growth.iterrows():
            date_str = row['date'].strftime('%Y-%m-%d')
            value_b = row['value_billions']
            qoq = row['qoq_growth_rate']
            yoy = row['yoy_growth_rate']

            qoq_str = f"{qoq:+.1f}%" if pd.notna(qoq) else "N/A"
            yoy_str = f"{yoy:+.1f}%" if pd.notna(yoy) else "N/A"

            print(f"{date_str}: ${value_b:.1f}B (QoQ: {qoq_str}, YoY: {yoy_str})")
    else:
        print("Forecast 데이터가 없습니다.")

    # 시각화
    plt.figure(figsize=(15, 12))

    # 값을 십억 단위로 변환
    df['sales_billions'] = df['value'] / 1e9
    growth_df['sales_billions'] = growth_df['value'] / 1e9

    # 실제 데이터와 예측 데이터 구분
    actual_data = df[df['date'] <= '2024-12-31']
    forecast_data = df[df['date'] > '2024-12-31']

    # 회사명은 티커 사용
    company_name = get_company_name(target_ticker)

    # 첫 번째 서브플롯: 매출 추이
    plt.subplot(2, 1, 1)

    if len(actual_data) > 0:
        plt.plot(actual_data['date'], actual_data['sales_billions'],
                 marker='o', linewidth=2, markersize=6, color='blue', label='실제 매출')

    if len(forecast_data) > 0:
        plt.plot(forecast_data['date'], forecast_data['sales_billions'],
                 marker='s', linewidth=2, markersize=6, color='red', linestyle='--', label='예측 매출')

    plt.title(f'{company_name} ({target_ticker}) 분기별 {target_item} 추이', fontsize=16, fontweight='bold')
    plt.xlabel('날짜', fontsize=12)
    plt.ylabel('매출 (십억 달러)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.xticks(rotation=45)

    # 두 번째 서브플롯: 성장률
    plt.subplot(2, 1, 2)

    # 성장률 데이터 (NaN이 아닌 값만)
    growth_data = growth_df.dropna(subset=['qoq_growth_rate', 'yoy_growth_rate'])

    if len(growth_data) > 0:
        plt.plot(growth_data['date'], growth_data['qoq_growth_rate'],
                 marker='o', linewidth=2, markersize=5, color='green', label='분기별 성장률 (QoQ)')
        plt.plot(growth_data['date'], growth_data['yoy_growth_rate'],
                 marker='s', linewidth=2, markersize=5, color='orange', label='연간 성장률 (YoY)')

    plt.title(f'{company_name} ({target_ticker}) {target_item} 성장률', fontsize=14, fontweight='bold')
    plt.xlabel('날짜', fontsize=12)
    plt.ylabel('성장률 (%)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.xticks(rotation=45)
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)  # 0% 라인 표시

    plt.tight_layout()
    plt.show()

    # 통계 정보 출력
    print(f"\n========== {target_item} 통계 ==========")
    max_value = df['value'].max()
    min_value = df['value'].min()
    max_date = df[df['value'] == max_value]['date'].iloc[0]
    min_date = df[df['value'] == min_value]['date'].iloc[0]

    print(f"최대 {target_item}: ${max_value/1e9:.3f}B ({max_date.strftime('%Y-%m-%d')})")
    print(f"최소 {target_item}: ${min_value/1e9:.3f}B ({min_date.strftime('%Y-%m-%d')})")
    print(f"평균 {target_item}: ${df['value'].mean()/1e9:.3f}B")

    # 연도별 매출 합계
    df['year'] = df['date'].dt.year
    yearly_sales = df.groupby('year')['value'].sum() / 1e9

    print(f"\n연도별 {target_item} 합계 (십억 달러):")
    for year, sales in yearly_sales.items():
        print(f"{year}: ${sales:.3f}B")

    # 성장률 통계
    if 'qoq_growth_rate' in growth_df.columns:
        # 전체 데이터의 성장률
        valid_qoq = growth_df['qoq_growth_rate'].dropna()
        valid_yoy = growth_df['yoy_growth_rate'].dropna()

        # Historical과 Forecast 데이터 구분 (2025년 6월 30일 기준)
        historical_growth = growth_df[growth_df['date'] <= '2025-06-30']
        forecast_growth = growth_df[growth_df['date'] > '2025-06-30']

        # Historical 성장률
        hist_qoq = historical_growth['qoq_growth_rate'].dropna()
        hist_yoy = historical_growth['yoy_growth_rate'].dropna()

        # Forecast 성장률
        forecast_qoq = forecast_growth['qoq_growth_rate'].dropna()
        forecast_yoy = forecast_growth['yoy_growth_rate'].dropna()

        if len(valid_qoq) > 0:
            print(f"\n========== 성장률 통계 ==========")
            print(f"[전체 기간]")
            print(f"평균 분기별 성장률: {valid_qoq.mean():.1f}%")
            print(f"평균 연간 성장률: {valid_yoy.mean():.1f}%")
            print(f"최고 분기별 성장률: {valid_qoq.max():.1f}%")
            print(f"최고 연간 성장률: {valid_yoy.max():.1f}%")

            if len(hist_qoq) > 0:
                print(f"\n[Historical 데이터 (2025년 6월까지)]")
                print(f"평균 분기별 성장률: {hist_qoq.mean():.1f}%")
                print(f"평균 연간 성장률: {hist_yoy.mean():.1f}%")

            if len(forecast_qoq) > 0:
                print(f"\n[예측 데이터 (2025년 7월 이후)]")
                print(f"평균 예상 분기별 성장률: {forecast_qoq.mean():.1f}%")
                print(f"평균 예상 연간 성장률: {forecast_yoy.mean():.1f}%")

else:
    print("데이터를 찾을 수 없습니다.")

# ==============================================
# 다른 티커나 아이템을 분석하고 싶을 때는 이 부분만 변경하세요
# ==============================================

print(f"\n========== 다른 분석 예시 ==========")
print("이제 어떤 티커든 빠르게 분석할 수 있습니다 (DB 재조회 없음):")
print("# 애플 분석")
print("df_aapl, growth_aapl = analyze_ticker_item('AAPL', 'sale', historical_df_full, forecast_df_full)")
print("# 구글 순이익 분석")
print("df_googl, growth_googl = analyze_ticker_item('GOOGL', 'netincome', historical_df_full, forecast_df_full)")
print("# 테슬라 분석")
print("df_tsla, growth_tsla = analyze_ticker_item('TSLA', 'sale', historical_df_full, forecast_df_full)")

# 실제 다른 회사 분석 예시 (주석 해제하여 사용)
# df_aapl, growth_aapl = analyze_ticker_item('AAPL', 'sale', historical_df_full, forecast_df_full)
# df_googl, growth_googl = analyze_ticker_item('GOOGL', 'netincome', historical_df_full, forecast_df_full)


========== NVDA - sale 분석 ==========


KeyError: 'ticker'

In [3]:
df

NameError: name 'df' is not defined

In [6]:
def calculate_forecast_growth_rates(forecast_df_full, target_item='sale'):
    """
    예측 데이터에서 향후 4분기, 8분기, 12분기 평균 성장률을 계산
    """

    # 특정 target_col에 해당하는 데이터만 필터링
    df_filtered = forecast_df_full[forecast_df_full['target_col'] == target_item].copy()

    if len(df_filtered) == 0:
        print(f"경고: {target_item}에 대한 예측 데이터가 없습니다.")
        return pd.DataFrame()

    # 날짜 컬럼을 datetime으로 변환
    df_filtered['forecast_date'] = pd.to_datetime(df_filtered['forecast_date'])

    growth_results = []

    for ticker in df_filtered['ticker'].unique():
        ticker_data = df_filtered[df_filtered['ticker'] == ticker].copy()

        # forecast_date로 정렬
        ticker_data = ticker_data.sort_values('forecast_date').reset_index(drop=True)

        if len(ticker_data) < 4:  # 최소 4분기 데이터 필요
            continue

        # 0 또는 음수 값이 있는 경우 성장률 계산에서 제외하거나 특별 처리
        ticker_data = ticker_data[ticker_data['forecast_value'] > 0]  # 양수 값만 사용

        if len(ticker_data) < 4:  # 양수 값이 4개 미만인 경우 스킵
            continue

        # 분기별 성장률 계산 (QoQ)
        ticker_data['qoq_growth_rate'] = ticker_data['forecast_value'].pct_change() * 100

        # inf 값과 NaN 값을 제거
        ticker_data['qoq_growth_rate'] = ticker_data['qoq_growth_rate'].replace([np.inf, -np.inf], np.nan)

        # 향후 4분기, 8분기, 12분기 평균 성장률 계산
        # NaN과 inf 값을 제외하고 계산
        valid_growth_rates = ticker_data['qoq_growth_rate'].dropna()
        valid_growth_rates = valid_growth_rates[np.isfinite(valid_growth_rates)]

        growth_4q = valid_growth_rates.iloc[:4].mean() if len(valid_growth_rates) >= 4 else np.nan
        growth_8q = valid_growth_rates.iloc[:8].mean() if len(valid_growth_rates) >= 8 else np.nan
        growth_12q = valid_growth_rates.iloc[:12].mean() if len(valid_growth_rates) >= 12 else np.nan

        # 첫 번째와 마지막 값으로 전체 기간 성장률도 계산
        if len(ticker_data) >= 2:
            first_val = ticker_data['forecast_value'].iloc[0]
            last_val = ticker_data['forecast_value'].iloc[-1]
            total_periods = len(ticker_data) - 1

            if first_val > 0 and last_val > 0 and total_periods > 0:
                total_growth_rate = ((last_val / first_val) ** (1/total_periods) - 1) * 100
                # inf 체크
                if not np.isfinite(total_growth_rate):
                    total_growth_rate = np.nan
            else:
                total_growth_rate = np.nan
        else:
            total_growth_rate = np.nan

        # 결과에 inf 값이 없는지 확인
        result = {
            'ticker': ticker,
            'target_item': target_item,
            'forecast_periods': len(ticker_data),
            'avg_qoq_growth_4q': growth_4q if np.isfinite(growth_4q) else np.nan,
            'avg_qoq_growth_8q': growth_8q if np.isfinite(growth_8q) else np.nan,
            'avg_qoq_growth_12q': growth_12q if np.isfinite(growth_12q) else np.nan,
            'total_cagr': total_growth_rate if np.isfinite(total_growth_rate) else np.nan,
            'first_forecast_value': ticker_data['forecast_value'].iloc[0],
            'latest_forecast_value': ticker_data['forecast_value'].iloc[-1],
            'first_forecast_date': ticker_data['forecast_date'].iloc[0],
            'last_forecast_date': ticker_data['forecast_date'].iloc[-1]
        }

        growth_results.append(result)

    return pd.DataFrame(growth_results)

def display_growth_rankings(growth_df, sort_by='avg_qoq_growth_4q', top_n=20):
    """
    성장률 랭킹을 표시하는 함수
    """

    if len(growth_df) == 0:
        print("표시할 데이터가 없습니다.")
        return

    # inf, -inf, NaN 값 제거하고 정렬
    valid_data = growth_df.copy()
    valid_data = valid_data[np.isfinite(valid_data[sort_by])]
    valid_data = valid_data.dropna(subset=[sort_by])

    if len(valid_data) == 0:
        print(f"{sort_by}에 대한 유효한 데이터가 없습니다.")
        return

    sorted_df = valid_data.sort_values(sort_by, ascending=False).head(top_n)

    print(f"\n========== {sort_by}로 정렬된 상위 {min(len(sorted_df), top_n)}개 기업 ==========")
    print(f"{'순위':<4} {'Ticker':<8} {'예측기간':<6} {'4Q평균':<8} {'8Q평균':<8} {'12Q평균':<8} {'전체CAGR':<8} {'최신예측값':<12}")
    print("-" * 80)

    for idx, (_, row) in enumerate(sorted_df.iterrows(), 1):
        # 각 값이 유한한지 확인하고 출력
        growth_4q = f"{row['avg_qoq_growth_4q']:>7.2f}%" if np.isfinite(row['avg_qoq_growth_4q']) else "    N/A"
        growth_8q = f"{row['avg_qoq_growth_8q']:>7.2f}%" if np.isfinite(row['avg_qoq_growth_8q']) else "    N/A"
        growth_12q = f"{row['avg_qoq_growth_12q']:>7.2f}%" if np.isfinite(row['avg_qoq_growth_12q']) else "    N/A"
        total_cagr = f"{row['total_cagr']:>7.2f}%" if np.isfinite(row['total_cagr']) else "    N/A"

        print(f"{idx:<4} {row['ticker']:<8} {int(row['forecast_periods']):<6} "
              f"{growth_4q} {growth_8q} {growth_12q} {total_cagr} "
              f"{row['latest_forecast_value']:>11,.0f}")

def analyze_all_items_growth(forecast_df_full, items_list=None):
    """
    여러 재무 항목에 대한 성장률 분석을 수행
    """

    if items_list is None:
        # 예측 데이터에서 사용 가능한 target_col 확인
        available_items = forecast_df_full['target_col'].unique()
        print(f"사용 가능한 재무 항목들: {list(available_items)}")
        items_list = list(available_items)  # 모든 항목 분석

    all_results = {}

    for item in items_list:
        print(f"\n{'='*20} {item.upper()} 분석 {'='*20}")

        growth_df = calculate_forecast_growth_rates(forecast_df_full, item)

        if len(growth_df) > 0:
            all_results[item] = growth_df

            # 각 성장률 지표별로 상위 랭킹 표시
            print(f"\n[{item}] 향후 4분기 평균 QoQ 성장률 순위:")
            display_growth_rankings(growth_df, 'avg_qoq_growth_4q', 15)

            print(f"\n[{item}] 향후 8분기 평균 QoQ 성장률 순위:")
            display_growth_rankings(growth_df, 'avg_qoq_growth_8q', 15)

            print(f"\n[{item}] 향후 12분기 평균 QoQ 성장률 순위:")
            display_growth_rankings(growth_df, 'avg_qoq_growth_12q', 15)

            print(f"\n[{item}] 전체 기간 CAGR 순위:")
            display_growth_rankings(growth_df, 'total_cagr', 15)

            # 통계 요약 (inf 값 제외)
            print(f"\n[{item}] 성장률 통계 요약:")
            print(f"분석 대상 기업 수: {len(growth_df)}")

            # 각 지표별로 유효한 데이터만으로 통계 계산
            for metric, name in [
                ('avg_qoq_growth_4q', '4분기 평균 성장률'),
                ('avg_qoq_growth_8q', '8분기 평균 성장률'),
                ('avg_qoq_growth_12q', '12분기 평균 성장률'),
                ('total_cagr', '전체 기간 CAGR')
            ]:
                valid_values = growth_df[metric]
                valid_values = valid_values[np.isfinite(valid_values)]
                valid_values = valid_values.dropna()

                if len(valid_values) > 0:
                    print(f"{name} - 평균: {valid_values.mean():.2f}%, "
                          f"중앙값: {valid_values.median():.2f}%, "
                          f"유효 데이터 수: {len(valid_values)}")

        else:
            print(f"{item}에 대한 데이터가 충분하지 않습니다.")

    return all_results

def get_top_growth_summary(results, metric='avg_qoq_growth_4q', top_n=5):
    """
    각 재무 항목별 최고 성장률 기업들의 요약을 보여주는 함수 (inf 값 제외)
    """
    print(f"\n{'='*50}")
    print(f"모든 재무 항목별 {metric} 기준 상위 {top_n}개 기업 요약")
    print(f"{'='*50}")

    for item, df in results.items():
        if len(df) > 0:
            # inf 값과 NaN 제거
            valid_df = df.copy()
            valid_df = valid_df[np.isfinite(valid_df[metric])]
            valid_df = valid_df.dropna(subset=[metric])

            if len(valid_df) > 0:
                top_companies = valid_df.nlargest(top_n, metric)
                print(f"\n[{item.upper()}] 상위 {min(len(top_companies), top_n)}개:")
                for idx, (_, row) in enumerate(top_companies.iterrows(), 1):
                    print(f"  {idx}. {row['ticker']}: {row[metric]:.2f}%")
            else:
                print(f"\n[{item.upper()}] 유효한 데이터가 없습니다.")

# ==============================================
# 메인 실행 부분
# ==============================================

print("예측 성장률 분석 시작...")

# DB 엔진 생성
engine = make_engine(db_info)

# 전체 예측 데이터 로드
print("전체 예측 데이터 로딩 중...")
try:
    sql = text("SELECT * FROM us_fs_forecast_data")
    with engine.connect() as conn:
        forecast_df_full = pd.read_sql(sql, conn)
    print(f"전체 Forecast 데이터 로드 완료: {len(forecast_df_full)} 행")
    print(f"사용 가능한 ticker 수: {forecast_df_full['ticker'].nunique()}")
    print(f"사용 가능한 재무 항목: {list(forecast_df_full['target_col'].unique())}")

    # inf 값 미리 제거
    numeric_columns = ['forecast_value', 'lower_ci', 'upper_ci']
    for col in numeric_columns:
        if col in forecast_df_full.columns:
            forecast_df_full[col] = forecast_df_full[col].replace([np.inf, -np.inf], np.nan)

    print(f"\ninf 값 제거 후 데이터: {len(forecast_df_full)} 행")

except Exception as e:
    print(f"예측 데이터 로드 오류: {e}")
    forecast_df_full = pd.DataFrame()

if len(forecast_df_full) > 0:
    # 모든 재무 항목에 대한 성장률 분석 수행
    results = analyze_all_items_growth(forecast_df_full)

    # 각 지표별 상위 기업 요약
    get_top_growth_summary(results, 'avg_qoq_growth_4q', 5)
    get_top_growth_summary(results, 'avg_qoq_growth_8q', 5)
    get_top_growth_summary(results, 'avg_qoq_growth_12q', 5)
    get_top_growth_summary(results, 'total_cagr', 5)

    # 결과를 Excel 파일로 저장 (inf 값 제거된 데이터로)
    try:
        with pd.ExcelWriter('forecast_growth_analysis.xlsx', engine='openpyxl') as writer:
            for item, df in results.items():
                # inf 값 제거하고 성장률 기준으로 정렬해서 저장
                clean_df = df.copy()
                for col in ['avg_qoq_growth_4q', 'avg_qoq_growth_8q', 'avg_qoq_growth_12q', 'total_cagr']:
                    clean_df[col] = clean_df[col].replace([np.inf, -np.inf], np.nan)

                sorted_df = clean_df.sort_values('avg_qoq_growth_4q', ascending=False, na_position='last')
                sorted_df.to_excel(writer, sheet_name=f'{item}_growth', index=False)
        print("\n분석 결과가 'forecast_growth_analysis.xlsx' 파일로 저장되었습니다.")
    except Exception as e:
        print(f"Excel 저장 오류: {e}")

else:
    print("예측 데이터를 로드할 수 없어 분석을 수행할 수 없습니다.")

예측 성장률 분석 시작...
전체 예측 데이터 로딩 중...
전체 Forecast 데이터 로드 완료: 84696 행
사용 가능한 ticker 수: 5607
사용 가능한 재무 항목: ['sale', 'opiti']

inf 값 제거 후 데이터: 84696 행
사용 가능한 재무 항목들: ['sale', 'opiti']

==================== SALE 분석 ====================

[sale] 향후 4분기 평균 QoQ 성장률 순위:

========== avg_qoq_growth_4q로 정렬된 상위 15개 기업 ==========
순위   Ticker   예측기간   4Q평균     8Q평균     12Q평균    전체CAGR   최신예측값       
--------------------------------------------------------------------------------
1    ANNX     24     49268396.05% 24634210.39% 24677648.68%    4.88%           3
2    PASG     12     19585365.94% 14680502.80%     N/A   10.51%           9
3    EWTX     12     5690328.05% 4267739.80%     N/A   10.51%          27
4    GPCR     12     4445325.91% 3387464.74%     N/A   11.52%          30
5    INAB     12     2555739.53% 1916563.40%     N/A   10.51%          41
6    PEPG     12     1530669.54% 1148078.57%     N/A   10.50%          69
7    CABA     12     1300375.30% 975181.40%     N/A   10.51%          52
8    KTTA

In [8]:
sorted_df['target_item'].unique().tolist()

['opiti']

In [9]:
df

,ticker,target_item,forecast_periods,avg_qoq_growth_4q,avg_qoq_growth_8q,avg_qoq_growth_12q,total_cagr,first_forecast_value,latest_forecast_value,first_forecast_date,last_forecast_date
0,A,opiti,12,1.145720,1.147064,NaN,1.148074,3.034321e+08,3.440292e+08,2025-09-30,2028-06-30
1,AA,opiti,12,57.620257,57.976545,NaN,7.170339,7.437438e+07,1.593103e+08,2025-09-30,2028-06-30
2,AAL,opiti,9,45.518862,38.749078,NaN,17.420804,3.662292e+08,1.323468e+09,2025-09-30,2028-06-30
3,AAN,opiti,12,0.000000,0.000000,NaN,0.000000,1.751659e+07,1.751659e+07,2025-09-30,2028-06-30
4,AAON,opiti,12,0.000000,0.000000,NaN,0.000000,2.358200e+07,2.358200e+07,2025-09-30,2028-06-30
...,...,...,...,...,...,...,...,...,...,...,...
424,BVH,opiti,12,1.336894,4.533075,NaN,-3.944558,1.231008e+07,7.906833e+06,2025-09-30,2028-06-30
425,BVN,opiti,12,4.312870,3.995631,NaN,3.936502,9.244378e+07,1.413601e+08,2025-09-30,2028-06-30
426,BVS,opiti,9,-18.552141,128.937607,NaN,-43.039036,1.607630e+07,1.781578e+05,2025-09-30,2028-03-31
427,BW,opiti,11,54.910064,38.456084,NaN,15.011619,1.212837e+07,4.911564e+07,2025-12-31,2028-06-30
